# B3 — decoder-ağırlıklı INR, 600 pair (v1, 16 Eki)
Paylaşılan **sinüslü decoder** (229K parametre, INT8 QAT) + pair başına **28'lik latent**. Baytların ~%90'ı decoder'da.

**Neden bu yol:** #95 (hnerv_muon, 0.199 → cila ile 0.188) birebir bu mimariyi kullandı; Eylül'deki `HNeRVDecoder`'ımızla aynı
(sin aktivasyon, kanallar [36, 36, 36, 27, 20, 18, 18], 28'lik latent, `0.1·sin(refine)`). Biz yolu 50–80 epoch'luk QAT koşularıyla kapattık;
#95 ~19.500 epoch QAT açık eğitti. Karar notu: `karar-0.2-icin-mantik-secimi-2026-10-16.md`.

**Kapı deneyi (§7):** kaynak `b3_kaynak_sin_fp32_0.1598.pt` (4 Eki `control_nomask/best.pt`, FP32 dist 0.1598: seg 0.1255, pose 0.0343) →
QAT8 + EMA + L7, 1500 epoch, her 100 epoch'ta 8 bit decoder + 8 bit latent ile ölçüm, en iyi **toplam**a göre.
Kıyas: r3_v2 (4 Eki, QAT E50) dist 0.188 / 207 KB / 0.326. **Karar:** E1500'de dist belirgin aşağıda ve LR→0'da hâlâ iniyorsa uzun koşu; platoya yapışıyorsa yol kapalı.

**Kurallar:** karar toplam skorla · 1 KB ≈ 0.00067 · sadece `SON` satırı okunur (kazanç LR→0'da gelir) · her koşu yeni etiket · eğitim ölçümle aynı hassasiyette (FP32, TF32 yalnız eğitimde).

**Çalıştırma:** §1–6 yalnız tanım (eğitim yok) → §7 deney hücreleri. Bölüm 1–3 ve 5, `b2_tezgah600_v6`'dan aynen; 4 ve 6 bu mimariye uyarlandı.

## 1. Kurulum

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| cihaz:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"| bellek: {props.total_memory / 2**30:.1f} GB")

ROOT = Path("/content/drive/MyDrive/deneme")
VIDEO_PATH = Path("/content/drive/MyDrive/0.mkv")
MODEL_DIR = ROOT / "models"
TARGET_CACHE = ROOT / "checkpoints" / "hnerv384_targets_evalrgb_v1.pt"
KAYNAK = ROOT / "b2" / "b3_kaynak_sin_fp32_0.1598.pt"   # 4 Eki control_nomask/best.pt kopyası (Drive'da b2 klasörüne kondu)
OUT_DIR = ROOT / "b3"                                     # bu notebook'un bütün çıktıları
OUT_DIR.mkdir(exist_ok=True)
LOG = OUT_DIR / "b3_log.txt"


def log(msg):
    # SON satırları elle kaydetmek insan hatasına açık: her şey hem ekrana hem dosyaya
    print(msg, flush=True)
    with open(LOG, "a") as f:
        f.write(msg + "\n")


for p in (VIDEO_PATH, MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", TARGET_CACHE, KAYNAK):
    print("OK " if p.exists() else "YOK", p)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130 | cihaz: cuda
GPU: NVIDIA A100-SXM4-40GB | bellek: 39.5 GB
OK  /content/drive/MyDrive/0.mkv
OK  /content/drive/MyDrive/deneme/models/posenet.safetensors
OK  /content/drive/MyDrive/deneme/models/segnet.safetensors
OK  /content/drive/MyDrive/deneme/checkpoints/hnerv384_targets_evalrgb_v1.pt
OK  /content/drive/MyDrive/deneme/b2/b3_kaynak_sin_fp32_0.1598.pt


In [ ]:
import gc, sys, traceback, types, warnings, torch, torch.nn as nn

KORU = {"distortion_net", "segnet", "posenet", "seg_targets", "pose_targets", "PILOT", "device"}  # silinmeyecekler

def _gpu_tutuyor(o, d=0):
    if torch.is_tensor(o): return o.is_cuda
    if isinstance(o, nn.Module): return any(t.is_cuda for t in [*o.parameters(), *o.buffers()])
    if isinstance(o, torch.optim.Optimizer):
        return any(torch.is_tensor(v) and v.is_cuda for s in o.state.values() for v in s.values()) \
            or any(p.is_cuda for g in o.param_groups for p in g["params"])
    if d < 2 and isinstance(o, dict): return any(_gpu_tutuyor(v, d + 1) for v in list(o.values())[:500])
    if d < 2 and isinstance(o, (list, tuple)): return any(_gpu_tutuyor(v, d + 1) for v in list(o)[:500])
    return False

def _kareleri_bosalt(exc, gorulen=None):
    gorulen = gorulen or set()
    while exc is not None and id(exc) not in gorulen:            # hata zinciri: cause / context
        gorulen.add(id(exc))
        if exc.__traceback__ is not None:
            traceback.clear_frames(exc.__traceback__)            # karelerdeki yerel değişkenleri sil
        _kareleri_bosalt(exc.__cause__, gorulen)
        exc = exc.__context__

def gpu_temizle(*isimler, koru=()):
    """gpu_temizle() → GPU tutan tüm globalleri sil (KORU hariç). İsim verirsen onları da siler."""
    try: torch.cuda.synchronize()                                # yarıda kalan GPU işini bitir
    except Exception: pass
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for o in gc.get_objects():                               # kim saklarsa saklasın: tüm traceback'ler
            try:
                if isinstance(o, types.TracebackType):
                    traceback.clear_frames(o)
            except ReferenceError:                               # ölü weakref proxy: dokununca patlıyor, atla
                continue
    ip = get_ipython()
    hatalar = [getattr(sys, "last_exc", None), getattr(sys, "last_value", None)]
    if getattr(ip, "last_execution_result", None) is not None:
        hatalar.append(ip.last_execution_result.error_in_exec)
        ip.last_execution_result.error_in_exec = None
    for e in hatalar:
        if isinstance(e, BaseException): _kareleri_bosalt(e)
    if getattr(sys, "last_traceback", None) is not None:
        traceback.clear_frames(sys.last_traceback)
    sys.last_exc = sys.last_type = sys.last_value = sys.last_traceback = None
    g = ip.user_ns
    sil = set(isimler) | {n for n, v in list(g.items())
                          if not n.startswith("_") and n not in KORU | set(koru) and _gpu_tutuyor(v)}
    for n in sorted(sil):
        g.pop(n, None)
    print("silindi:", ", ".join(sorted(sil)) or "—")
    ip.run_line_magic("reset", "-f out")
    gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()


def gpu_durum(etiket=""):
    a, r = torch.cuda.memory_allocated() / 2**30, torch.cuda.memory_reserved() / 2**30
    t = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"[GPU {etiket}] kullanılan {a:.1f} GB | ayrılmış {r:.1f} GB | toplam {t:.1f} GB")

## 2. Değerlendiriciler ve hedefler (SegNet / PoseNet donmuş)

In [ ]:

!uv pip -q install segmentation-models-pytorch timm einops safetensors av

import sys
sys.path.insert(0, str(ROOT))                      # research/ klasörü Drive'da, deneme/ altında
from research.modules import DistortionNet
from research.oracle_pair import seg_preprocess_grad, pose_preprocess_grad

distortion_net = DistortionNet().eval().to(device)
distortion_net.load_state_dicts(MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", device)
distortion_net.requires_grad_(False)               # ağırlıkları asla değişmeyecek

n_seg = sum(p.numel() for p in distortion_net.segnet.parameters())
n_pose = sum(p.numel() for p in distortion_net.posenet.parameters())
print(f"SegNet {n_seg / 1e6:.1f}M | PoseNet {n_pose / 1e6:.1f}M parametre (donmuş)")

SegNet 9.5M | PoseNet 13.9M parametre (donmuş)


In [ ]:
targets = torch.load(TARGET_CACHE, map_location="cpu", weights_only=True)
seg_targets = targets["seg_class"].to(device, dtype=torch.uint8)    # [600, 384, 512]: piksel başına sınıf
pose_targets = targets["pose"].to(device, dtype=torch.float32)      # [600, 6]: pair başına kamera hareketi
NUM_PAIRS = seg_targets.shape[0]
print("Seg:", tuple(seg_targets.shape), "| Pose:", tuple(pose_targets.shape), "| pair:", NUM_PAIRS)

Seg: (600, 384, 512) | Pose: (600, 6) | pair: 600


## 3. Skor (resmi formül, FP32, TF32 kapalı)

In [ ]:




import torch.nn.functional as F

def inflate(native):
    """[B, 2, 3, 384, 512] → [B, 2, 3, 874, 1164]. İleri geçişte yuvarlar, geri geçişte gradyanı geçirir."""
    b = native.shape[0]
    up = F.interpolate(native.flatten(0, 1), size=(874, 1164), mode="bicubic", align_corners=False)
    up = up.clamp(0, 255)
    up = up + (up.round() - up).detach()          # STE: ileri = round, geri = özdeşlik
    return up.view(b, 2, 3, 874, 1164)


@torch.no_grad()
def score(render, ids_all, bs=16):
    """render(ids) → [B, 2, 3, 384, 512] (0–255). Resmi ölçümle aynı ayar: FP32, TF32 kapalı."""
    old = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    wrong, pix, pose_sq = 0, 0, 0.0
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        full = inflate(render(ids))
        pred = distortion_net.segnet(seg_preprocess_grad(full)).argmax(1)
        pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
        wrong += int((pred != seg_targets[ids].long()).sum())
        pix += pred.numel()
        pose_sq += float((pose - pose_targets[ids]).square().mean(1).sum())
    torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = old
    seg, pose_t = 100 * wrong / pix, (10 * pose_sq / len(ids_all)) ** 0.5
    return {"dist": seg + pose_t, "seg": seg, "pose": pose_t}


PILOT = torch.arange(600, device=device)         # tam video; 120 tezgâhı istenirse arange(120)

## 4. Model: sinüslü decoder + pair başına latent
`HNeRVDecoder` Eylül'deki sınıfın kendisi (alpha karışımı atıldı; alpha=0 ile birebir aynı). `B3` sarmalayıcısı latentleri
`codes` adıyla tutar ki `score(m, PILOT)` ve v6 araçları (`EMA`, `qat_ac`) doğrudan çalışsın.

In [ ]:
import math
import torch.nn as nn


class HNeRVDecoder(nn.Module):
    """= #95 model.py: sin aktivasyon, kanallar [36, 36, 36, 27, 20, 18, 18], Conv→PixelShuffle + bilinear skip, 0.1·sin(refine)."""
    def __init__(self, latent_dim=28, base_channels=36):
        super().__init__()
        c = base_channels
        self.channels = [c, c, c, int(c * 0.75), int(c * 0.58), int(c * 0.5), int(c * 0.5)]
        self.stem = nn.Linear(latent_dim, self.channels[0] * 6 * 8)
        self.blocks = nn.ModuleList()
        self.skips = nn.ModuleList()
        for i in range(6):
            cin, cout = self.channels[i], self.channels[i + 1]
            self.blocks.append(nn.Conv2d(cin, cout * 4, 3, padding=1))
            self.skips.append(nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity())
        self.ps = nn.PixelShuffle(2)
        f = self.channels[-1]
        self.refine = nn.Sequential(nn.Conv2d(f, f // 2, 3, padding=2, dilation=2),
                                    nn.Conv2d(f // 2, f, 3, padding=1))
        self.rgb_0 = nn.Conv2d(f, 3, 3, padding=1)
        self.rgb_1 = nn.Conv2d(f, 3, 3, padding=1)

    def forward(self, z):
        x = torch.sin(self.stem(z).view(len(z), self.channels[0], 6, 8))
        for block, skip in zip(self.blocks, self.skips):
            identity = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            x = torch.sin(self.ps(block(x)) + skip(identity))
        x = x + 0.1 * torch.sin(self.refine(x))
        f0 = torch.sigmoid(self.rgb_0(x)) * 255.0
        f1 = torch.sigmoid(self.rgb_1(x)) * 255.0
        return torch.stack([f0, f1], dim=1)


class B3(nn.Module):
    """codes [600, 28] + paylaşılan decoder. forward(ids) → [B, 2, 3, 384, 512]."""
    def __init__(self, n_pairs, latent_dim=28, base_channels=36):
        super().__init__()
        self.codes = nn.Parameter(torch.zeros(n_pairs, latent_dim))
        self.dec = HNeRVDecoder(latent_dim, base_channels)

    def forward(self, ids, codes=None):
        z = codes if codes is not None else self.codes[ids]
        return self.dec(z)


def kaynak_yukle(yol):
    """Eylül formatı {'decoder': state_dict, 'latents': [600, 28]} → B3. b3 formatı {'model': state_dict} de kabul."""
    ck = torch.load(yol, map_location="cpu", weights_only=False)
    if "model" in ck:
        m = B3(NUM_PAIRS, ck["model"]["codes"].shape[1]).to(device)
        m.load_state_dict(ck["model"]); return m
    m = B3(ck["latents"].shape[0], ck["latents"].shape[1]).to(device)
    m.dec.load_state_dict(ck["decoder"])
    with torch.no_grad():
        m.codes.copy_(ck["latents"])
    return m


n_dec = sum(p.numel() for p in HNeRVDecoder().parameters())
print(f"decoder {n_dec:,} parametre | latent {NUM_PAIRS}×28 = {NUM_PAIRS * 28:,} değer")

decoder 228,958 parametre | latent 600×28 = 16,800 değer


## 5. Kayıp (L7 + pose) — #95 ile aynı

In [ ]:
L7 = dict(tau=0.3, thr=1.0, mult=4.0)

def l7_loss(logits, truth):
  correct = logits.gather(1, truth.unsqueeze(1))
  others = logits.clone()
  others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
  margin = correct - others.amax(1, keepdim=True)
  per_pixel = L7["tau"] * F.softplus(-margin / L7["tau"])
  with torch.no_grad():
    w = 1.0 + L7["mult"] * (margin < L7["thr"]).float()
    w = w / w.mean()
  return (per_pixel * w).mean()


def task_loss(native, ids):
  full = inflate(native)
  logits = distortion_net.segnet(seg_preprocess_grad(full))
  pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
  seg_l = 100 * l7_loss(logits, seg_targets[ids].long())
  pose_l = torch.sqrt(10 * F.mse_loss(pose, pose_targets[ids]) + 1e-6)
  return seg_l + pose_l, seg_l.item(), pose_l.item()

## 6. Nicemleme, boyut ve ölçüm araçları
v6'dan fark: decoder'da `nn.Linear` var (stem 28→1728, **48K parametre, ağın %21'i**); QAT ve KB sayımı Conv2d'ye ek olarak Linear'ı da kapsar.
Ölçek: Conv kanal başına, Linear tensör başına (1728 satır başına fp16 ölçek 3.4 KB yük olurdu). Latentler boyut başına simetrik 8 bit.
`olc(m)` = cetvel: arşivdeki gibi (8 bit decoder + 8 bit latent) ölçer ve **sayı döndürür**.

In [ ]:
def qmax_of(bits):
    return int(round(2 ** (bits - 1) - 1))

In [ ]:
import copy, time
from tqdm.auto import tqdm

rate_of_kb = lambda kb: 25 * kb * 1e3 / 37545489  # KB → skor

KUCUK = ("skips", "refine", "rgb")     # bu adla başlayan katmanlar her zaman 8 bit; () → eski davranış


def katman_bit(ad, bits):
    return 8 if ad.startswith(KUCUK) else bits

def _olcek(w, qmax):
    if w.dim() == 4:
        return w.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax   # Conv: kanal başına
    return w.abs().amax().clamp_min(1e-12) / qmax                                   # Linear: tensör başına


@torch.no_grad()
def quant_net_(m, bits=8):
    """Decoder ağırlıklarını yerinde simetrik yuvarlar; küçük katmanlar katman_bit ile 8 bitte kalır."""
    for ad, mod in m.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            qmax = qmax_of(katman_bit(ad, bits))
            s = _olcek(mod.weight, qmax)
            mod.weight.copy_(torch.round(mod.weight / s).clamp(-qmax, qmax) * s)



def qat_ac(m, bits=8):
    """İleri/geri geçiş yuvarlanmış ağırlığı görür, adım FP32 ağırlığa uygulanır (STE). Dönen sözlüğü backward'dan SONRA qat_kapat'a ver."""
    asil = {}
    for ad, mod in m.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            qmax = qmax_of(katman_bit(ad, bits))
            w = mod.weight.data
            s = _olcek(w, qmax)
            asil[ad] = w
            mod.weight.data = torch.round(w / s).clamp(-qmax, qmax) * s
    return asil


def qat_kapat(m, asil):
    for ad, mod in m.named_modules():
        if ad in asil:
            mod.weight.data = asil[ad]


def entropy_kb(q):
    """Tamsayı dizisinin ampirik entropisi (değerler bağımsız) → KB."""
    _, counts = torch.unique(q, return_counts=True)
    p = counts.double() / q.numel()
    return float(-(counts * p.log2()).sum()) / 8 / 1e3


@torch.no_grad()
def decoder_kb(dec, bits=8):
    """Gerçek format: ağırlıklar katman bitinde entropi kodlu, ölçekler ve bias'lar fp16."""
    kb = 0.0
    for ad, mod in dec.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            qmax = qmax_of(katman_bit(ad, bits))
            s = _olcek(mod.weight, qmax)
            kb += entropy_kb(torch.round(mod.weight / s).clamp(-qmax, qmax)) + 2 * s.numel() / 1e3
            if mod.bias is not None:
                kb += 2 * mod.bias.numel() / 1e3
    return kb


def kod_kb(codes, bits=8):
    """Latentler boyut başına simetrik `bits` bit (her boyutun aralığı farklı, pair'ler arası aynı). Döner: yuvarlanmış, KB."""
    qmax = qmax_of(bits)
    s = codes.abs().amax(dim=0, keepdim=True).clamp_min(1e-12) / qmax
    q = torch.round(codes / s).clamp(-qmax, qmax)
    return q * s, entropy_kb(q) + 2 * s.numel() / 1e3


@torch.no_grad()
def olc(m, bits=8, kod_bits = 8):
    """Arşivdeki gibi ölçer: decoder `bits` bit + latentler `bits` bit. Döner: dist, seg, pose, dec_kb, kod_kb, toplam."""
    em = copy.deepcopy(m)
    quant_net_(em.dec, bits)
    q, kkb = kod_kb(em.codes.data, kod_bits)
    em.codes.data = q
    r = score(em, PILOT)
    dkb = decoder_kb(em.dec, bits)
    del em; torch.cuda.empty_cache()
    return r | {"dec_kb": dkb, "kod_kb": kkb, "toplam": r["dist"] + rate_of_kb(dkb + kkb)}


class EMA:
    """Ağırlıkların hareketli ortalaması (#95: 0.999). ufuk_adim: gölgenin kabaca kaç adımı hatırladığı (decay = 1 - 1/ufuk)."""
    def __init__(self, model, ufuk_adim):
        self.decay = 1 - 1 / max(1, ufuk_adim)
        self.golge = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.n = 0

    @torch.no_grad()
    def guncelle(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))          # başta hızlı ısınma
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.golge[k].lerp_(v, 1 - d)
            else:
                self.golge[k].copy_(v)

    def kopya(self, sablon):
        m = copy.deepcopy(sablon)
        m.load_state_dict(self.golge)
        return m

In [ ]:
def izgara_tarama(yol, bitler=(8, 7, 6.5, 6, 5.5, 5)):
    """Bütün decoder aynı bitte, eğitimsiz; latent 8 bit. Eğim = 8 bite göre KB başına dist kaybı, fiyatla (0.00067) kıyaslanır."""
    m = kaynak_yukle(yol)
    taban = None
    for b in bitler:
        q = olc(m, b, 8)
        if taban is None:
            taban = q
            log(f"[ızgara] {b:>4} bit | dist {q['dist']:.4f} (seg {q['seg']:.4f}, pose {q['pose']:.4f}) "
                f"| decoder {q['dec_kb']:6.1f} KB | toplam {q['toplam']:.3f} | taban")
            continue
        dd = q["dist"] - taban["dist"]
        dkb = taban["dec_kb"] - q["dec_kb"]
        verim = dd / rate_of_kb(dkb)
        log(f"[ızgara] {b:>4} bit | dist {q['dist']:.4f} (seg {q['seg']:.4f}, pose {q['pose']:.4f}) "
            f"| decoder {q['dec_kb']:6.1f} KB | toplam {q['toplam']:.3f} | verim {verim:5.2f}")
    del m; torch.cuda.empty_cache()

## 7. Deneyler
### 7.0 Cetvel kontrolü (dakikalar): kaynak FP32 ve eğitimsiz 8 bit
Beklenen: FP32 dist **0.1598** (seg 0.1255, pose 0.0343). 8 bit PTQ, 4 Ekim'de 0.167 / 261 KB (brotli) idi; bizim entropi kodlu formatla KB daha düşük çıkmalı.
Bu hücre `olc`'un ve yüklemenin doğru olduğunu kanıtlar; ilk eğitim hücresinden önce çalıştırılır.

In [ ]:
m = kaynak_yukle(KAYNAK)
r = score(m, PILOT)
log(f"[kaynak] FP32 | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f})")
q = olc(m, 8)
log(f"[kaynak] 8 bit PTQ | dist {q['dist']:.4f} (seg {q['seg']:.4f}, pose {q['pose']:.4f}) "
    f"| decoder {q['dec_kb']:.1f} + latent {q['kod_kb']:.1f} KB | TOPLAM {q['toplam']:.3f}")
del m; torch.cuda.empty_cache()

[kaynak] FP32 | dist 0.1598 (seg 0.1255, pose 0.0343)
[kaynak] 8 bit PTQ | dist 0.1685 (seg 0.1324, pose 0.0361) | decoder 224.4 + latent 20.4 KB | TOPLAM 0.331


### 7.1 Kapı deneyi — `b3_egit` (birlikte yazılacak)
Amaç: v6 `net_ft_q`'nun bu mimariye uyarlanmış hali, ama **latentler de eğitilir** (#95 gibi, ayrı ve ~10× yüksek LR).

Fonksiyonun yapması gerekenler, sırayla:
1. `kaynak_yukle(init)` → model; parametre grupları: decoder (`m.dec`) ve `m.codes` ayrı LR.
2. Her adımda `asil = qat_ac(m.dec, bits)` → `task_loss(m(ids), ids)` → `backward` → `qat_kapat(m.dec, asil)` → grad clip 1.0 → `opt.step()` → `sched.step()` → `ema.guncelle(m)`.
3. LR: ısınma (`warm` epoch) + kosinüs → 0. Kalite LR→0'da gelir; ara satırlar yanıltır.
4. Her `eval_every` epoch'ta `olc(ema.kopya(m) ya da m, bits)`; **toplam**a göre en iyi checkpoint tutulur; `log(...)` ile satır yazılır.
5. Sonda `SON` satırı + `torch.save({"model": en_iyi_sd, "bits": bits}, OUT_DIR / f"b3_{tag}.pt")`.

Birlikte karar verilecekler: batch ve LR'ler (#95: batch 8, geç aşamada decoder 3e-5, latent 3e-4; v6: batch 40, 6.7e-4 oturmuş ağ için fazla yüksekti),
EMA açık mı (#95 açık, bizde 120'de etkisizdi), latentler eğitimde yuvarlansın mı (#95: hayır, eğitim sonrası 8 bit).

In [ ]:
# b3_egit(init, tag, epochs=1500, batch=20, lr_dec=..., lr_kod=..., bits=8, warm=10, ema_ep=None, eval_every=100)


In [ ]:
def b3_egit(init, tag, lr_dec=1e-4, lr_kod=1e-3, epochs=1500, batch=20, bits=8, kod_bits=8,
            warm=10, ema_ep=None, eval_every=100):
    m = kaynak_yukle(init)
    log(f"[{tag}] başladı | init {Path(init).name} | bits {bits} | kod_bits {kod_bits} | kucuk {KUCUK} "
        f"| lr {lr_dec:.0e}/{lr_kod:.0e} | ema {ema_ep} | {epochs} epoch")
    opt = torch.optim.Adam([{"params": m.dec.parameters(), "lr": lr_dec},
                            {"params": [m.codes],           "lr": lr_kod}])
    adim = math.ceil(len(PILOT) / batch)

    def carpan(s):
        isinma = min(1, (s + 1) / (warm * adim))
        kosinus = 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim))))
        return isinma * kosinus

    sched = torch.optim.lr_scheduler.LambdaLR(opt, carpan)
    ema = EMA(m, ema_ep * adim) if ema_ep else None

    en_iyi = {"toplam": float("inf")}
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            asil = qat_ac(m.dec, bits)
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            qat_kapat(m.dec, asil)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else m
            q = olc(em, bits, kod_bits)
            if q["toplam"] < en_iyi["toplam"]:
                en_iyi = q | {"ep": ep + 1, "sd": copy.deepcopy(em.state_dict())}
            log(f"[{tag}] E{ep + 1:04d} | {bits} bit dist {q['dist']:.4f} (seg {q['seg']:.4f}, pose {q['pose']:.4f}) "
                f"| decoder {q['dec_kb']:.1f} + latent {q['kod_kb']:.1f} KB | toplam {q['toplam']:.3f} "
                f"| en iyi {en_iyi['toplam']:.3f} (E{en_iyi['ep']})")
    log(f"[{tag}] SON | en iyi E{en_iyi['ep']} | {bits} bit dist {en_iyi['dist']:.4f} (seg {en_iyi['seg']:.4f}, "
        f"pose {en_iyi['pose']:.4f}) | decoder {en_iyi['dec_kb']:.1f} + latent {en_iyi['kod_kb']:.1f} KB "
        f"| TOPLAM {en_iyi['toplam']:.3f}")
    torch.save({"model": en_iyi["sd"], "bits": bits, "kod_bits": kod_bits, "kucuk": KUCUK},
               OUT_DIR / f"b3_{tag}.pt")
    return en_iyi

In [ ]:
def katman_duyarlilik(yol, taban,  hedef):
  m = kaynak_yukle(yol)
  d0 = olc(m, taban, 8)["dist"]

  for ad, mod, in m.dec.named_modules():
    if not isinstance(mod, (nn.Conv2d, nn.Linear)):
      continue
    em = copy.deepcopy(m)
    for ad2, mod2 in em.dec.named_modules():
      if isinstance(mod2, (nn.Conv2d, nn.Linear)):
          quant_net_(mod2, hedef if ad2 == ad else taban)
    q, _ = kod_kb(em.codes.data, 8)
    em.codes.data = q
    d = score(em, PILOT)["dist"]
    kb_kazanc = decoder_kb(mod, taban) - decoder_kb(mod, hedef)
    dd = d - d0
    verim = dd / rate_of_kb(kb_kazanc)
    log(f"{ad:12s} | Δdist {dd:+.4f} | {kb_kazanc:5.1f} KB | verim {verim:5.2f}")
    del em
    torch.cuda.empty_cache()

In [ ]:
BUDAMA_HARIC = ("skips", "refine", "rgb")     # küçük ve kırılgan (A1): budanmaz


@torch.no_grad()
def budama_uygula_(dec, oran):
    """Her ağırlıklı katmanda |w|'si en küçük `oran` kadarını sıfırlar. Döner: katman başına maske (eğitimde tekrar uygulamak için)."""
    maskeler = {}
    for ad, mod in dec.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)) and not ad.startswith(BUDAMA_HARIC):
            esik = torch.quantile(mod.weight.abs().flatten(), oran)
            maske = (mod.weight.abs() > esik).float()
            mod.weight.mul_(maske)
            maskeler[ad] = maske
    return maskeler


def budama_egrisi(yol, bits=7, oranlar=(0, 0.1, 0.2, 0.3, 0.4, 0.5)):
    """Eğitimsiz: budama + `bits` bit decoder + 8 bit latent. verim = Δdist / rate_of_kb(ΔKB), bütünü budamasıza göre."""
    m = kaynak_yukle(yol)
    taban = None
    for p in oranlar:
        em = copy.deepcopy(m)
        if p > 0:
            budama_uygula_(em.dec, p)
        q = olc(em, bits, 8)
        sifir = sum((mod.weight == 0).sum().item() for mod in em.dec.modules()
                    if isinstance(mod, (nn.Conv2d, nn.Linear)))
        toplam_w = sum(mod.weight.numel() for mod in em.dec.modules()
                       if isinstance(mod, (nn.Conv2d, nn.Linear)))
        satir = (f"[budama] %{100 * p:>3.0f} | sıfır %{100 * sifir / toplam_w:4.1f} | dist {q['dist']:.4f} "
                 f"(seg {q['seg']:.4f}, pose {q['pose']:.4f}) | decoder {q['dec_kb']:6.1f} KB | toplam {q['toplam']:.3f}")
        if taban is None:
            taban = q
            log(satir + " | taban")
        else:
            verim = (q["dist"] - taban["dist"]) / rate_of_kb(taban["dec_kb"] - q["dec_kb"])
            log(satir + f" | verim {verim:5.2f}")
        del em; torch.cuda.empty_cache()
    del m; torch.cuda.empty_cache()

In [ ]:
def b3_egit(init, tag, lr_dec=1e-4, lr_kod=1e-3, epochs=1500, batch=20, bits=8, kod_bits=8,
            warm=10, ema_ep=None, eval_every=100):
    m = kaynak_yukle(init)
    log(f"[{tag}] başladı | init {Path(init).name} | bits {bits} | kod_bits {kod_bits} | kucuk {KUCUK} "
        f"| lr {lr_dec:.0e}/{lr_kod:.0e} | ema {ema_ep} | {epochs} epoch")
    opt = torch.optim.Adam([{"params": m.dec.parameters(), "lr": lr_dec},
                            {"params": [m.codes],           "lr": lr_kod}])
    adim = math.ceil(len(PILOT) / batch)

    def carpan(s):
        isinma = min(1, (s + 1) / (warm * adim))
        kosinus = 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim))))
        return isinma * kosinus

    sched = torch.optim.lr_scheduler.LambdaLR(opt, carpan)
    ema = EMA(m, ema_ep * adim) if ema_ep else None

    en_iyi = {"toplam": float("inf")}
    kayip_top, seg_top, pose_top, n_adim = 0.0, 0.0, 0.0, 0
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            asil = qat_ac(m.dec, bits)
            loss, seg_l, pose_l = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            qat_kapat(m.dec, asil)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
            if ema: ema.guncelle(m)
            kayip_top += loss.item(); seg_top += seg_l; pose_top += pose_l; n_adim += 1
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            # vekil kayıp düşerken skor kötüleşiyorsa sorun L7–argmax ayrışması; canlı EMA'dan iyiyse sorun EMA
            egitim = f"eğitim kaybı {kayip_top / n_adim:.4f} (seg {seg_top / n_adim:.4f}, pose {pose_top / n_adim:.4f})"
            kayip_top, seg_top, pose_top, n_adim = 0.0, 0.0, 0.0, 0
            adaylar = [("canlı", m)] + ([("ema", ema.kopya(m))] if ema else [])
            olcumler = []
            for ad, model in adaylar:
                q = olc(model, bits, kod_bits)
                olcumler.append((ad, q))
                if q["toplam"] < en_iyi["toplam"]:
                    en_iyi = q | {"ep": ep + 1, "sd": copy.deepcopy(model.state_dict()), "kaynak": ad}
            lr_su_an = opt.param_groups[0]["lr"]
            log(f"[{tag}] E{ep + 1:04d} | lr {lr_su_an:.1e} | {egitim}")
            for ad, q in olcumler:
                log(f"[{tag}]        {ad:5s} | {bits} bit dist {q['dist']:.4f} (seg {q['seg']:.4f}, pose {q['pose']:.4f}) "
                    f"| decoder {q['dec_kb']:.1f} + latent {q['kod_kb']:.1f} KB | toplam {q['toplam']:.3f}")
            log(f"[{tag}]        en iyi {en_iyi['toplam']:.3f} (E{en_iyi['ep']}, {en_iyi['kaynak']})")
    log(f"[{tag}] SON | en iyi E{en_iyi['ep']} ({en_iyi['kaynak']}) | {bits} bit dist {en_iyi['dist']:.4f} "
        f"(seg {en_iyi['seg']:.4f}, pose {en_iyi['pose']:.4f}) | decoder {en_iyi['dec_kb']:.1f} "
        f"+ latent {en_iyi['kod_kb']:.1f} KB | TOPLAM {en_iyi['toplam']:.3f}")
    torch.save({"model": en_iyi["sd"], "bits": bits, "kod_bits": kod_bits, "kucuk": KUCUK},
               OUT_DIR / f"b3_{tag}.pt")
    return en_iyi

In [ ]:
import os, math

def durum_kaydet(yol, **durum):
    gecici = yol.with_suffix(".tmp")
    torch.save(durum, gecici)
    os.replace(gecici, yol)   # kesinti yazmanın ortasına denk gelirse yarım dosya kalmasın: ya eski ya yeni

In [ ]:
def b3_egit(init, tag, lr_dec=1e-4, lr_kod=1e-3, epochs=1500, batch=20, bits=8, kod_bits=8,
            warm=10, ema_ep=None, eval_every=100, lam=0.0):
    durum_yol = OUT_DIR / f"b3_{tag}_durum.pt"
    m = kaynak_yukle(init)
    opt = torch.optim.Adam([{"params": m.dec.parameters(), "lr": lr_dec},
                            {"params": [m.codes], "lr": lr_kod}])
    adim = math.ceil(NUM_PAIRS / batch)

    def carpan(s):
        return min(1, (s + 1) / (warm * adim)) * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim))))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, carpan)
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    en_iyi = {"toplam": float("inf")}
    bas = 1

    if durum_yol.exists():
        d = torch.load(durum_yol, map_location=device, weights_only=False)
        m.load_state_dict(d["model"])
        opt.load_state_dict(d["opt"])
        sched.load_state_dict(d["sched"])
        if ema:
            ema.golge, ema.n = d["ema_golge"], d["ema_n"]
        en_iyi, bas = d["en_iyi"], d["epoch"] + 1
        log(f"[{tag}] devam: E{bas}'den | şimdiye kadar en iyi {en_iyi['toplam']:.3f} (E{en_iyi['epoch']}, {en_iyi['kaynak']})")
    else:
        log(f"[{tag}] başla: {init} | lr {lr_dec:g}/{lr_kod:g} | {bits} bit, latent {kod_bits} | "
            f"epoch {epochs}, batch {batch}, ema {ema_ep}, küçük {KUCUK}")

    top = seg_t = pose_t = kb_t = 0.0
    n = 0
    for ep in range(bas, epochs + 1):
        for ids in torch.randperm(NUM_PAIRS, device=device).split(batch):
            opt.zero_grad(set_to_none=True)
            if lam:
                kb_y = yumusak_kb(m.dec, bits)
                (lam * rate_of_kb(kb_y)).backward()   # on FP32 weights, before qat_ac swaps them for rounded ones
                kb_t += kb_y.item()
            asil = qat_ac(m.dec, bits)
            loss, s_l, p_l = task_loss(m(ids), ids)
            loss.backward()
            qat_kapat(m.dec, asil)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
            if ema:
                ema.guncelle(m)
            top += loss.item(); seg_t += s_l; pose_t += p_l; n += 1






        if ep % eval_every and ep != epochs:
            continue

        log(f"[{tag}] E{ep:04d} | lr {sched.get_last_lr()[0]:.1e} | "
            f"eğitim kaybı {top / n:.4f} (seg {seg_t / n:.4f}, pose {pose_t / n:.4f})"
            + (f" | kb~ {kb_t / n:.1f}" if lam else ""))
        top = seg_t = pose_t = kb_t = 0.0
        n = 0

        adaylar = [("canlı", m)] + ([("ema", ema.kopya(m))] if ema else [])
        for kaynak, aday in adaylar:
            r = olc(aday, bits, kod_bits)
            log(f"[{tag}]        {kaynak:5s} | {bits} bit dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) "
                f"| decoder {r['dec_kb']:.1f} + latent {r['kod_kb']:.1f} KB | toplam {r['toplam']:.3f}")
            if r["toplam"] < en_iyi["toplam"]:
                en_iyi = r | {"epoch": ep, "kaynak": kaynak, "sd": copy.deepcopy(aday.state_dict())}
        log(f"[{tag}]        en iyi {en_iyi['toplam']:.3f} (E{en_iyi['epoch']}, {en_iyi['kaynak']})")

        durum_kaydet(durum_yol, model=m.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                     ema_golge=ema.golge if ema else None, ema_n=ema.n if ema else 0,
                     en_iyi=en_iyi, epoch=ep)

    log(f"[{tag}] SON | en iyi E{en_iyi['epoch']} ({en_iyi['kaynak']}) | {bits} bit dist {en_iyi['dist']:.4f} "
        f"(seg {en_iyi['seg']:.4f}, pose {en_iyi['pose']:.4f}) | decoder {en_iyi['dec_kb']:.1f} + "
        f"latent {en_iyi['kod_kb']:.1f} KB | TOPLAM {en_iyi['toplam']:.3f}")
    log(f"[{tag}] başla: {init} | lr {lr_dec:g}/{lr_kod:g} | {bits} bit, latent {kod_bits} | "
        f"epoch {epochs}, batch {batch}, ema {ema_ep}, küçük {KUCUK}, λ {lam}")
    torch.save({"model": en_iyi["sd"], "bits": bits, "kod_bits": kod_bits, "kucuk": KUCUK},
               OUT_DIR / f"b3_{tag}.pt")
    durum_yol.unlink()   # koşu bitti; aynı etiketle tekrar çalıştırılırsa eski durumdan devam etmesin
    return en_iyi

In [ ]:
def yumusak_kb(dec, bits):
    kb = 0.0
    for ad, mod in dec.named_modules():
        if not isinstance(mod, (nn.Conv2d, nn.Linear)):
            continue
        qmax = qmax_of(katman_bit(ad, bits))
        s = _olcek(mod.weight.detach(), qmax)   # scale held fixed: otherwise the penalty could cheat by moving the max weight to stretch the grid
        x = (mod.weight / s).clamp(-qmax, qmax) + qmax
        lo = x.detach().floor().clamp(max=2 * qmax - 1)
        fr = (x - lo).flatten()
        lo = lo.long().flatten()
        sayim = torch.zeros(2 * qmax + 1, device=x.device)
        sayim = sayim.index_add(0, lo, 1 - fr).index_add(0, lo + 1, fr)
        p = sayim / fr.numel()
        kb = kb - (sayim * torch.log2(p + 1e-12)).sum() / 8 / 1e3
    return kb

In [ ]:
KUCUK = ()
_ = b3_egit(OUT_DIR / "b3_ent5b.pt", "hlr", lr_dec=1e-4, lr_kod=1e-3, bits=7, kod_bits=8,
            ema_ep=25, epochs=1500, eval_every=1, lam=5.0)

[hlr] devam: E301'den | şimdiye kadar en iyi 0.251 (E100, ema)
[hlr] E0301 | lr 1.0e-04 | eğitim kaybı 0.4533 (seg 0.4098, pose 0.0434) | kb~ 169.4
[hlr] SON | en iyi E100 (ema) | 7 bit dist 0.1247 (seg 0.0904, pose 0.0343) | decoder 170.2 + latent 20.0 KB | TOPLAM 0.251
[hlr] başla: /content/drive/MyDrive/deneme/b3/b3_ent5b.pt | lr 0.0001/0.001 | 7 bit, latent 8 | epoch 1500, batch 20, ema 25, küçük (), λ 5.0


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
olc(kaynak_yukle(OUT_DIR / "b3_ent5b.pt"), bits=7, kod_bits=8)

{'dist': 0.11311224609421107,
 'seg': 0.08299424913194445,
 'pose': 0.030117996962266618,
 'dec_kb': 169.93260691275455,
 'kod_kb': 20.00825740339355,
 'toplam': 0.23958607116288183}

In [ ]:
KUCUK = ()
_ = b3_egit(OUT_DIR / "b3_ent5b.pt", "hlr0", lr_dec=1e-4, lr_kod=1e-3, bits=7, kod_bits=8,
            ema_ep=50, epochs=1500, eval_every=100, lam=0.0)

[hlr0] başla: /content/drive/MyDrive/deneme/b3/b3_ent5b.pt | lr 0.0001/0.001 | 7 bit, latent 8 | epoch 1500, batch 20, ema 50, küçük ()
[hlr0] E0100 | lr 9.9e-05 | eğitim kaybı 0.4763 (seg 0.4277, pose 0.0486)
[hlr0]        canlı | 7 bit dist 0.1631 (seg 0.1137, pose 0.0494) | decoder 186.1 + latent 20.0 KB | toplam 0.300
[hlr0]        ema   | 7 bit dist 0.1191 (seg 0.0857, pose 0.0333) | decoder 184.7 + latent 20.0 KB | toplam 0.255
[hlr0]        en iyi 0.255 (E100, ema)
[hlr0] E0200 | lr 9.6e-05 | eğitim kaybı 0.5011 (seg 0.4479, pose 0.0532)
[hlr0]        canlı | 7 bit dist 0.1698 (seg 0.1188, pose 0.0510) | decoder 190.0 + latent 20.0 KB | toplam 0.310
[hlr0]        ema   | 7 bit dist 0.1193 (seg 0.0862, pose 0.0331) | decoder 189.1 + latent 20.0 KB | toplam 0.259
[hlr0]        en iyi 0.255 (E100, ema)
[hlr0] E0300 | lr 9.0e-05 | eğitim kaybı 0.5045 (seg 0.4505, pose 0.0540)
[hlr0]        canlı | 7 bit dist 0.1707 (seg 0.1168, pose 0.0539) | decoder 191.7 + latent 20.0 KB | toplam 

In [ ]:
from google.colab import runtime
runtime.unassign()